# 30. Two player extraction: what is in it, and how often did the old pipeline pick the other player?

`two_player/extract_two_players.py` ran the same detector, tracker and pose model as the main pipeline (YOLOv8m, ByteTrack, RTMPose-m) but kept **both players** of every clip: person 0 is the player nearer the camera, person 1 the one further away, fixed by geometry (larger mean bottom edge of the box).
It also logged which player the old area rule would have picked. Nothing in `keypoints_interp` is touched; everything new goes to `keypoints_two_*` folders.

This notebook does five things.
1. Merge the shard logs and check every clip is there.
2. Describe the extraction: how many clips have two players, how big the second player is, how confident the poses are.
3. **Compare with the skeleton the old pipeline actually used** (`keypoints_interp`), by where the player's hips are. This is the direct measure of how often the old pipeline used the far player, and it also reflects the later manual and wrist-speed corrections, which the geometric log column does not.
4. Apply the same smoothing and repair as the old pipeline to both players and save a new two person dataset, `keypoints_two_interp`, shape (2, 30, 17, 3) per clip.
5. Show clips with both players drawn, so you can judge by eye **who hits the shot** in a random sample. That gives the real rate of the far player being the actor.

**About the A/B letter in the file name.** It is recorded in the log but not used for any choice. The error audit showed it does not separate accurate from inaccurate clips, and a visual check showed it only sometimes matches the far player.
It is shown below next to the geometry so the write-up can state what was tried.

In [1]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.signal import savgol_filter
from clearml import Task

try:
    import cv2
except ImportError:
    cv2 = None

REPO = os.environ.get("BADMINTON_REPO", "/workspace/badminton-honours-project")
DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
RAW_VIDEOS = os.environ.get("BADMINTON_RAW_VIDEOS", f"{DATA}/raw")
EXPORT = f"{OUTPUTS}/results"
os.makedirs(EXPORT, exist_ok=True)
sys.path.insert(0, REPO)
import dataset as ds

TWO_RAW, TWO_BOX = f"{DATA}/keypoints_two_raw", f"{DATA}/boxes_two"
TWO_INTERP = f"{DATA}/keypoints_two_interp"          # new, written by this notebook
OLD_INTERP = f"{DATA}/keypoints_interp"              # old single player data, read only
N_TARGET, SG_WINDOW, SG_POLY, CONF_MIN = 30, 5, 2, 0.3
NEAR, FAR = "#4c72b0", "#dd8452"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

task = Task.init(project_name="BadmintonPoseAnalysis/results", task_name="two_player_check", task_type=Task.TaskTypes.data_processing,
                 tags=["analysis", "two_player"], reuse_last_task_id=False, auto_connect_frameworks={"matplotlib": False})
logger = task.get_logger()

ModuleNotFoundError: No module named 'pandas'

## 1. Merge the logs and check coverage

In [ ]:
def read_csvs(paths):
    # a run with no failures leaves a failures file with no columns, which pandas refuses to read
    frames = []
    for p in paths:
        try:
            frames.append(pd.read_csv(p))
        except pd.errors.EmptyDataError:
            print(f"{os.path.basename(p)} is empty (no rows)")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


log_files = sorted(f for f in glob.glob(f"{DATA}/extraction_two_log*.csv") if "merged" not in f)
print("logs:", [os.path.basename(f) for f in log_files])
two = read_csvs(log_files)
print("rows before removing repeats:", len(two))
two = two.drop_duplicates("filepath", keep="last")           # the 20 clip smoke run is in the unsuffixed log
fail_files = glob.glob(f"{DATA}/extraction_two_failures*.csv")
failures = read_csvs(fail_files)
print("failures:", len(failures))
if len(failures):
    display(failures.head(10))

manifest = ds.load_filtered_manifest(f"{DATA}/split_manifest.csv", f"{DATA}/extraction_log.csv",
                                     f"{DATA}/keypoints_interp_log.csv", min_native_frames=5)
all_manifest = pd.read_csv(f"{DATA}/split_manifest.csv")
missing = sorted(set(all_manifest["filepath"]) - set(two.loc[two["status"] == "ok", "filepath"]))
print(f"clips in the manifest: {len(all_manifest)}; with two player output: {int((two['status'] == 'ok').sum())}; missing: {len(missing)}")
if missing:
    print(missing[:10])
two = two[two["status"] == "ok"].set_index("filepath")
two.reset_index().to_csv(f"{DATA}/extraction_two_log_merged.csv", index=False)

## 2. What was extracted

In [ ]:
t = two.copy()
t["old_extraction_method"] = pd.read_csv(f"{DATA}/extraction_log.csv").set_index("filepath")["selection_method"].reindex(t.index)
print("clips by number of players found:")
print(t["n_persons"].value_counts().sort_index().to_string())

summary = pd.DataFrame({
    "near player area (px^2)": t["person0_area"].describe(),
    "far player area (px^2)": t.loc[t["n_persons"] == 2, "person1_area"].describe(),
    "far / near area": t.loc[t["n_persons"] == 2, "area_ratio_second_first"].describe(),
    "near mean confidence": t["person0_mean_conf"].describe(),
    "far mean confidence": t.loc[t["n_persons"] == 2, "person1_mean_conf"].describe(),
    "near coverage": t["person0_coverage"].describe(),
    "far coverage": t.loc[t["n_persons"] == 2, "person1_coverage"].describe(),
}).round(3)
display(summary)
by_split = pd.crosstab(t["split"], t["n_persons"], normalize="index").round(3)
by_class = pd.crosstab(t["class_name"].str.split("_", n=1).str[1], t["n_persons"], normalize="index").round(2)
display(by_split)
display(by_class)
logger.report_table("extraction summary", "two_player", iteration=0, table_plot=summary.reset_index())
logger.report_table("persons found by class", "two_player", iteration=0, table_plot=by_class.reset_index())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(t.loc[t["n_persons"] == 2, "area_ratio_second_first"], bins=40, color="#8172b3")
axes[0].set_xlabel("far player area / near player area"); axes[0].set_ylabel("clips")
axes[1].hist([t["person0_mean_conf"], t.loc[t["n_persons"] == 2, "person1_mean_conf"]], bins=30, color=[NEAR, FAR], label=["near", "far"])
axes[1].set_xlabel("mean keypoint confidence"); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Which player did the old pipeline actually use?

For every clip the old skeleton (`keypoints_interp`) is compared with the new near and far skeletons by the median hip position over the frames where both hips are visible. The old skeleton is assigned to whichever new player it sits closer to.
This reflects what was really used, including the later manual review and wrist speed corrections. If the closest distance is large, neither new skeleton matches and the clip is reported separately.

In [ ]:
def median_hip(kp):
    hips = kp[:, [11, 12], :]
    ok = (hips[:, :, 2] > CONF_MIN).all(axis=1)
    if not ok.any():
        return np.array([np.nan, np.nan])
    return np.median(hips[ok, :, :2].mean(axis=1), axis=0)


rows = []
for fp, r in two.iterrows():
    stem = os.path.splitext(os.path.basename(fp))[0]
    new = np.load(f"{TWO_RAW}/{r['class_name']}/{stem}.npy")
    boxes = np.load(f"{TWO_BOX}/{r['class_name']}/{stem}.npy")
    old = np.load(f"{OLD_INTERP}/{r['class_name']}/{stem}.npy")
    h_old = median_hip(old)
    h = [median_hip(new[p][~np.isnan(boxes[p, :, 0])]) if (~np.isnan(boxes[p, :, 0])).any() else np.array([np.nan, np.nan]) for p in (0, 1)]
    d = [np.linalg.norm(h_old - h[p]) for p in (0, 1)]
    rows.append({"filepath": fp, "dist_near": d[0], "dist_far": d[1]})
cmp_ = pd.DataFrame(rows).set_index("filepath")
cmp_["closest (px)"] = cmp_[["dist_near", "dist_far"]].min(axis=1)
cmp_["old skeleton is"] = np.where(cmp_["dist_near"].fillna(np.inf) <= cmp_["dist_far"].fillna(np.inf), "near", "far")
cmp_.loc[cmp_["closest (px)"].isna(), "old skeleton is"] = "unknown"
cmp_["matched"] = cmp_["closest (px)"] < 60            # px; a hip centre moves less than this between two tracker runs of the same player
cmp_["letter"] = two["actor_letter"]
cmp_["split"] = two["split"]
cmp_["old method"] = t["old_extraction_method"]
cmp_["old geometric rule picked"] = two["legacy_person"].map({0: "near", 1: "far", -1: "unknown"})
cmp_["n_persons"] = two["n_persons"]

print("distance from the old skeleton to the closest new skeleton (px):")
print(cmp_["closest (px)"].describe().round(1).to_string())
print(f"\nold skeleton matches one of the new players within 60 px: {cmp_['matched'].mean():.1%}")
m = cmp_[cmp_["matched"]]
print(f"among matched clips, the old pipeline used the FAR player in {(m['old skeleton is'] == 'far').mean():.1%} ({(m['old skeleton is'] == 'far').sum()} of {len(m)})")

In [ ]:
m = cmp_[cmp_["matched"] & (cmp_["n_persons"] == 2)]
tables = {
    "split": pd.crosstab(m["split"], m["old skeleton is"], normalize="index").round(3).assign(clips=m["split"].value_counts()),
    "old selection method": pd.crosstab(m["old method"], m["old skeleton is"], normalize="index").round(3).assign(clips=m["old method"].value_counts()),
    "letter in file name": pd.crosstab(m["letter"], m["old skeleton is"], normalize="index").round(3).assign(clips=m["letter"].value_counts()),
    "class": pd.crosstab(two.loc[m.index, "class_name"].str.split("_", n=1).str[1], m["old skeleton is"], normalize="index").round(2)
                .assign(clips=two.loc[m.index, "class_name"].str.split("_", n=1).str[1].value_counts()),
}
for name, tab in tables.items():
    print(f"share of clips where the old skeleton was the near / far player, by {name}")
    display(tab)
    logger.report_table(f"old skeleton by {name}", "two_player", iteration=0, table_plot=tab.reset_index())

agree = pd.crosstab(m["old geometric rule picked"], m["old skeleton is"])
print("geometric rule recorded in the new log (what the area rule alone would pick) against the skeleton really used:")
display(agree)

How to read it: the table by old selection method shows whether the later corrections (`wrist_tiebreak`, `manual_review`, `area_reextracted`) moved the choice toward the far player. If most clips are near whatever the method, the old pipeline used the near player almost everywhere.
Whether that was wrong depends on who actually hits the shot, which the visual sample at the end of this notebook measures.

## 4. Build the two person dataset

The same cleaning as `repair_resampling.py`, applied to each player separately, on the 30 frames saved by the two player extraction. Those 30 frames are sampled across the span of the clip where the players are tracked; a clip with fewer than 30 frames in that span has repeated frames,
which are removed first, then the real frames are smoothed (Savitzky-Golay, window 5, order 2, on x and y) and interpolated back to 30. A player missing from some frames has those frames filled by linear interpolation of x and y and **confidence set to 0**, so a model can see that the frame is not real.
A player who is not found at all is stored as zeros with confidence 0. The output is never written over an existing folder.

In [ ]:
def standardize_indices(n_available, n_target=N_TARGET):
    return np.linspace(0, n_available - 1, n_target).round().astype(int)


def smooth(seq):
    if seq.shape[0] < SG_WINDOW:
        return seq.copy()
    out = seq.copy()
    out[:, :, :2] = savgol_filter(seq[:, :, :2], SG_WINDOW, SG_POLY, axis=0)
    return out


def resample_linear(seq, n_target=N_TARGET):
    n = seq.shape[0]
    if n == 1:
        return np.repeat(seq, n_target, axis=0)
    src, dst = np.arange(n), np.linspace(0, n - 1, n_target)
    flat = seq.reshape(n, -1)
    out = np.stack([np.interp(dst, src, flat[:, i]) for i in range(flat.shape[1])], axis=1)
    return out.reshape((n_target,) + seq.shape[1:]).astype(seq.dtype)


def clean_person(raw, present, span_frames):
    """raw (30, 17, 3), present (30,) bool, span_frames = real frames the 30 slots were sampled from."""
    n = int(min(span_frames, N_TARGET))
    if n < N_TARGET:
        idx = standardize_indices(n)
        first_slot = [int(np.where(idx == k)[0][0]) for k in range(n)]
        native, pres = raw[first_slot].astype(np.float32).copy(), present[first_slot]
    else:
        native, pres = raw.astype(np.float32).copy(), present.copy()
    if not pres.any():
        return np.zeros((N_TARGET, 17, 3), dtype=np.float32), n, 0.0
    if (~pres).any():
        fr = np.arange(n)
        for j in range(17):
            for c in range(2):
                native[~pres, j, c] = np.interp(fr[~pres], fr[pres], native[pres, j, c])
        native[~pres, :, 2] = 0.0
    out = smooth(native)
    if n < N_TARGET:
        out = resample_linear(out)
    out[:, :, 2] = np.clip(out[:, :, 2], 0.0, 1.0)
    return out.astype(np.float32), n, float(pres.mean())


# self test on made up data: a clip with 12 real frames, one player missing in two of them
rng = np.random.RandomState(0)
true = rng.rand(12, 17, 3).astype(np.float32)
slots = standardize_indices(12)
fake_raw = true[slots]
fake_present = np.ones(N_TARGET, dtype=bool)
fake_present[np.isin(slots, [3, 4])] = False
fake_raw[~fake_present] = 0
out, n, cov = clean_person(fake_raw, fake_present, 12)
assert out.shape == (N_TARGET, 17, 3) and n == 12 and 0 < cov < 1
assert np.all(np.diff(slots) >= 0) and (out[:, :, 2] >= 0).all() and (out[:, :, 2] <= 1).all()
out0, _, cov0 = clean_person(fake_raw * 0, np.zeros(N_TARGET, dtype=bool), 12)
assert cov0 == 0 and not out0.any()
print("cleaning self test passed")

In [ ]:
if os.path.exists(TWO_INTERP):
    print(f"{TWO_INTERP} already exists; not rebuilding. Delete it first if you mean to.")
    interp_log = pd.read_csv(f"{DATA}/keypoints_two_interp_log.csv")
else:
    rows = []
    for fp, r in two.iterrows():
        stem = os.path.splitext(os.path.basename(fp))[0]
        raw = np.load(f"{TWO_RAW}/{r['class_name']}/{stem}.npy")
        boxes = np.load(f"{TWO_BOX}/{r['class_name']}/{stem}.npy")
        both, ns, covs = [], [], []
        for p in (0, 1):
            out, n, cov = clean_person(raw[p], ~np.isnan(boxes[p, :, 0]), int(r["span_frames"]))
            both.append(out); ns.append(n); covs.append(cov)
        os.makedirs(f"{TWO_INTERP}/{r['class_name']}", exist_ok=True)
        np.save(f"{TWO_INTERP}/{r['class_name']}/{stem}.npy", np.stack(both).astype(np.float32))
        rows.append({"filepath": fp, "n_native": ns[0], "n_persons": int(r["n_persons"]),
                     "near_present": covs[0], "far_present": covs[1]})
    interp_log = pd.DataFrame(rows)
    interp_log.to_csv(f"{DATA}/keypoints_two_interp_log.csv", index=False)
    print(f"wrote {len(interp_log)} clips to {TWO_INTERP}")

print("real frames per clip (n_native):")
print(interp_log["n_native"].describe().round(1).to_string())
print(f"clips with fewer than 5 real frames: {(interp_log['n_native'] < 5).sum()}")
chk = np.load(f"{TWO_INTERP}/{two['class_name'].iloc[0]}/{os.path.splitext(os.path.basename(two.index[0]))[0]}.npy")
print("shape check:", chk.shape, chk.dtype, "finite:", bool(np.isfinite(chk).all()))

## 5. Look at both players

Blue is the player nearer the camera (person 0), orange the one further away (person 1); a box is drawn from the tracker for each. Frames are the real video frames at the positions the extraction sampled.
Set `BADMINTON_RAW_VIDEOS` to the folder holding the source clips if the default `/workspace/data/raw` is wrong.

In [ ]:
def draw_person(ax, kp, box, colour):
    if box is None or np.isnan(box[0]):
        return
    vis = kp[:, 2] > CONF_MIN
    for a, b in ds.COCO_SKELETON:
        if vis[a] and vis[b]:
            ax.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], color=colour, lw=1.6)
    ax.scatter(kp[vis, 0], kp[vis, 1], s=7, color=colour, zorder=3)
    ax.add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec=colour, lw=1.1, ls="--"))


def show_two(fp, tag="", frames=(0, 7, 14, 21, 29)):
    r = two.loc[fp]
    stem = os.path.splitext(os.path.basename(fp))[0]
    kps = np.load(f"{TWO_RAW}/{r['class_name']}/{stem}.npy")
    boxes = np.load(f"{TWO_BOX}/{r['class_name']}/{stem}.npy")
    targets = np.linspace(r["span_first"], r["span_last"], N_TARGET).round().astype(int)
    path = f"{RAW_VIDEOS}/{fp}"
    cap = cv2.VideoCapture(path) if (cv2 is not None and os.path.exists(path)) else None

    ok_boxes = boxes[~np.isnan(boxes[:, :, 0])]
    x0, y0 = ok_boxes[:, [0, 1]].min(axis=0)
    x1, y1 = ok_boxes[:, [2, 3]].max(axis=0)
    mx, my = 0.08 * (x1 - x0), 0.08 * (y1 - y0)
    fig, axes = plt.subplots(1, len(frames), figsize=(2.9 * len(frames), 3.4))
    for ax, t in zip(axes, frames):
        if cap is not None:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(targets[t]))
            ok, bgr = cap.read()
            if ok:
                ax.imshow(bgr[..., ::-1])
        draw_person(ax, kps[0, t], boxes[0, t], NEAR)
        draw_person(ax, kps[1, t], boxes[1, t], FAR)
        ax.set_xlim(x0 - mx, x1 + mx); ax.set_ylim(y1 + my, y0 - my)
        ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"frame {t}", fontsize=7)
    if cap is not None:
        cap.release()
    old = cmp_.loc[fp, "old skeleton is"] if fp in cmp_.index else "?"
    cls = r["class_name"].split("_", 1)[1]
    fig.suptitle(f"{tag} {cls}   |   old pipeline used: {old}   letter {r['actor_letter']}   persons found {int(r['n_persons'])}   "
                 f"(blue = near, orange = far)\n{stem}", fontsize=8, y=1.06)
    plt.show()


if cv2 is None:
    print("cv2 is not installed: the clips below will show skeletons on a blank canvas")
elif not os.path.isdir(RAW_VIDEOS):
    print(f"no video folder at {RAW_VIDEOS}: the clips below will show skeletons on a blank canvas")

### Random sample: who hits the shot?

30 clips drawn at random over all splits (seed 0, so the same clips every run). For each, decide who performs the shot: the **near** player (blue), the **far** player (orange), **both** (an exchange inside the clip), or **unclear**.
Fill the dictionary in the next cell using the number in each title. This gives the rate at which the far player is the actor, and combined with section 3, the rate at which the old pipeline used the wrong player.

In [ ]:
N_SAMPLE = 30
sample = two[two["n_persons"] == 2].sample(N_SAMPLE, random_state=0).index.tolist()
for i, fp in enumerate(sample[:15]):
    show_two(fp, tag=f"#{i}")

In [ ]:
for i, fp in enumerate(sample[15:], start=15):
    show_two(fp, tag=f"#{i}")

In [ ]:
# Fill in after looking: who performs the shot? "near" = blue, "far" = orange, "both", or "unclear".
ACTOR = {
    # 0: "near",
    # 1: "far",
    near, far, far, near, near
}

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half


verdicts = pd.DataFrame({"number": range(len(sample)), "filepath": sample})
verdicts["actor"] = verdicts["number"].map(ACTOR)
verdicts["old skeleton is"] = cmp_.reindex(sample)["old skeleton is"].to_numpy()
verdicts.to_csv(f"{EXPORT}/two_player_actor_verdicts.csv", index=False)

judged = verdicts.dropna(subset=["actor"])
print(f"judged {len(judged)} of {len(verdicts)} clips")
if len(judged):
    print(judged["actor"].value_counts().to_string())
    nf = judged[judged["actor"].isin(["near", "far"])]
    lo, hi = wilson(int((nf["actor"] == "far").sum()), len(nf))
    print(f"\nfar player performs the shot: {(nf['actor'] == 'far').mean():.0%} of {len(nf)} clear clips (95% interval {lo:.0%} to {hi:.0%})")
    wrong = nf[nf["old skeleton is"].isin(["near", "far"]) & (nf["old skeleton is"] != nf["actor"])]
    usable = nf[nf["old skeleton is"].isin(["near", "far"])]
    lo, hi = wilson(len(wrong), len(usable))
    print(f"old pipeline used the wrong player: {len(wrong)} of {len(usable)} = {len(wrong) / max(len(usable), 1):.0%} (95% interval {lo:.0%} to {hi:.0%})")
    print(pd.crosstab(nf["actor"], nf["old skeleton is"]).to_string())

### Clips where the old pipeline used the far player

If section 3 found any, these are the clips to check next: was the far player the actor there (the later corrections worked), or not?

In [ ]:
far_used = cmp_[(cmp_["old skeleton is"] == "far") & cmp_["matched"] & (cmp_["n_persons"] == 2)]
print(f"{len(far_used)} clips; showing 8 at random")
for fp in far_used.sample(min(8, len(far_used)), random_state=0).index:
    show_two(fp, tag="far used")

### Clips where only one player was found

For these the second player is zeros with confidence 0. Check whether a second player is really absent (a close-up, a player out of frame) or was missed.

In [ ]:
one = two[two["n_persons"] < 2]
print(f"{len(one)} clips with fewer than two players ({len(one) / len(two):.1%})")
for fp in one.sample(min(6, len(one)), random_state=0).index:
    show_two(fp, tag="one player")

In [ ]:
# Is the "far" person plausibly the opponent? Depth gap = how much further up the frame the far player's feet are,
# as a fraction of the near player's height. Bystanders at the same depth have a gap near 0.
rows = []
for fp, r in two.iterrows():
    stem = os.path.splitext(os.path.basename(fp))[0]
    boxes = np.load(f"{TWO_BOX}/{r['class_name']}/{stem}.npy")
    f = {"filepath": fp}
    for p in (0, 1):
        ok = ~np.isnan(boxes[p, :, 0])
        if ok.any():
            b = boxes[p][ok]
            centres = (b[:, :2] + b[:, 2:]) / 2
            f[f"h{p}"] = np.median(b[:, 3] - b[:, 1])
            f[f"foot{p}"] = np.median(b[:, 3])
            f[f"cx{p}"] = np.median(centres[:, 0])
            f[f"travel{p}"] = (np.linalg.norm(np.diff(centres, axis=0), axis=1).sum() / f[f"h{p}"]) if len(b) > 1 else 0.0
    rows.append(f)
far = pd.DataFrame(rows).set_index("filepath")
far["n_persons"] = two["n_persons"]
far["depth gap"] = (far["foot0"] - far["foot1"]) / far["h0"]
far["size ratio"] = far["h1"] / far["h0"]
far["sideways separation"] = (far["cx0"] - far["cx1"]).abs() / far["h0"]
two2 = far[far["n_persons"] == 2]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes[:3], ["depth gap", "size ratio", "sideways separation"]):
    ax.hist(two2[col].clip(upper=two2[col].quantile(0.995)), bins=50, color="#8172b3"); ax.set_xlabel(col)
axes[3].scatter(two2["sideways separation"], two2["depth gap"], s=4, alpha=0.4)
axes[3].set_xlabel("sideways separation"); axes[3].set_ylabel("depth gap")
plt.tight_layout(); plt.show()
display(two2[["depth gap", "size ratio", "sideways separation", "travel0", "travel1"]].describe().round(2))

# thresholds: set these from the histograms (a second hump near 0 is the bystanders), then look at the clips below
GAP_MIN, SIZE_MAX = 0.25, 1.0
far["far_valid"] = (far["n_persons"] == 2) & (far["depth gap"] >= GAP_MIN) & (far["size ratio"] <= SIZE_MAX)
print(f"far player passes the check in {far['far_valid'].sum()} of {len(far)} clips ({far['far_valid'].mean():.1%})")
print("clips that fail, by number of candidate tracks the detector saw:")
print(far[~far["far_valid"] & (far["n_persons"] == 2)].join(two["n_candidate_tracks"], rsuffix="_")["n_candidate_tracks"].value_counts().sort_index().to_string())
far[["depth gap", "size ratio", "sideways separation", "travel0", "travel1", "far_valid"]].to_csv(f"{DATA}/two_player_far_valid.csv")

print("\nfailing the check (should look like bystanders):")
for fp in far[~far["far_valid"] & (far["n_persons"] == 2)].sample(6, random_state=0).index:
    show_two(fp, tag="FAILS")
print("\njust passing the check (should look like opponents):")
edge = far[far["far_valid"] & (far["depth gap"] < GAP_MIN + 0.1)]
for fp in edge.sample(min(6, len(edge)), random_state=0).index:
    show_two(fp, tag="PASSES")

In [ ]:
DRILL = "2023-05-02"
src = pd.Series(np.where(two.index.str.contains(DRILL), "drill session", "match"), index=two.index, name="source")
d = far.join(src).join(two[["n_candidate_tracks", "person1_mean_conf", "class_name", "split"]]).join(cmp_[["old skeleton is"]])
d["class"] = d["class_name"].str.split("_", n=1).str[1]

summary = d.groupby("source").agg(
    clips=("n_persons", "size"), candidate_tracks=("n_candidate_tracks", "median"),
    far_confidence=("person1_mean_conf", "median"), sideways_separation=("sideways separation", "median"),
    far_travel=("travel1", "median"), near_travel=("travel0", "median"),
    old_pipeline_used_far=("old skeleton is", lambda s: (s == "far").mean())).round(2)
display(summary)

comp = pd.crosstab(d["class"], d["source"])
comp["drill share"] = (comp["drill session"] / comp.sum(axis=1)).round(2)
display(comp.sort_values("drill share", ascending=False))
logger.report_table("two player by source", "two_player", iteration=0, table_plot=summary.reset_index())

# 12 match and 12 drill clips, to judge who hits the shot in each
two_ok = d[d["n_persons"] == 2]
tags = {f"M{i}": fp for i, fp in enumerate(two_ok[two_ok["source"] == "match"].sample(12, random_state=1).index)}
tags |= {f"D{i}": fp for i, fp in enumerate(two_ok[two_ok["source"] == "drill session"].sample(12, random_state=1).index)}
for tag, fp in tags.items():
    show_two(fp, tag=tag)

In [ ]:
# who performs the shot? "near" = blue, "far" = orange, "both", "unclear", or "bystander" if the orange person is not a player
ACTOR2 = {
    far, near, far, near, near, near, near, near, far, 
}
rows = [{"tag": t, "source": "match" if t[0] == "M" else "drill", "actor": ACTOR2.get(t), "old": d.loc[fp, "old skeleton is"]} for t, fp in tags.items()]
v = pd.DataFrame(rows); v.to_csv(f"{EXPORT}/two_player_actor_verdicts_by_source.csv", index=False)
for s, g in v.dropna(subset=["actor"]).groupby("source"):
    print(s, g["actor"].value_counts().to_dict())
    nf = g[g["actor"].isin(["near", "far"])]
    lo, hi = wilson(int((nf["actor"] == "far").sum()), len(nf))
    print(f"   far performs the shot: {(nf['actor'] == 'far').mean():.0%} of {len(nf)} (95% interval {lo:.0%} to {hi:.0%})")

In [ ]:
import glob
import numpy as np, cv2, matplotlib.pyplot as plt
part = "2022-08-30_18-18-31_dataset_set1_046_004321_004347"
z = np.load(glob.glob(f"/workspace/data/tracks_all/*/{part}*.npz")[0])
rows, n = z["rows"], int(z["n_frames"])
cap = cv2.VideoCapture(glob.glob(f"/workspace/data/raw/*/{part}*")[0])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, f in zip(axes, (0, n // 2, n - 1)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, bgr = cap.read()
    if ok: ax.imshow(bgr[..., ::-1])
    for tid, fr, x1, y1, x2, y2, c in rows[rows[:, 1] == f]:
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=plt.cm.tab10(int(tid) % 10), lw=1.5))
        ax.text(x1, y1 - 3, f"id {int(tid)} ({c:.2f})", color=plt.cm.tab10(int(tid) % 10), fontsize=7)
    ax.set_title(f"frame {f}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("tracks:", sorted(set(rows[:, 0].astype(int))))

### What to take from this notebook

- **Section 3** is the rate at which the old pipeline used the far player. If it is near zero, the old pipeline used the near player almost always.
- **The random sample** is the rate at which the far player is the actor. If the far player performs the shot in a sizeable share of clips, then the old skeleton was the wrong player in roughly that share, and a two person model can recover it.
- If the far player almost never performs the shot, the wrong player problem seen in the error audit is rarer than it looked, and the second player's value is context (where the opponent stands) rather than a fix.
- The next notebook trains ST-GCN on the new data: near only, far only, and both players, on validation only with the tuning seeds.